# TravelPlan playground

Quick starter for poking at `travelplanner.travelplan`. Run cells top-to-bottom; each one is independent enough to re-run.

If imports fail, run `uv sync` in this directory first so the editable `travelplanner` install is picked up.

In [1]:
from datetime import date, datetime

from travelplanner.travelplan import (
    Day,
    DayNotFoundError,
    Slot,
    SlotNotFoundError,
    SlotOverlapError,
    TravelPlan,
)


def slot(name, start, end, **kwargs):
    """Tiny helper so test data stays readable."""
    return Slot(
        name=name,
        start_time=datetime.fromisoformat(start),
        end_time=datetime.fromisoformat(end),
        **kwargs,
    )

## 1. Build a sample 2-day plan

In [2]:
plan = TravelPlan(title="Rome 2-day demo")
plan.add_day(label="Arrival", calendar_date=date(2026, 6, 1))
plan.add_day(calendar_date=date(2026, 6, 2))

# Day 1
plan.add_slot(1, slot(
    "Breakfast", "2026-06-01T08:00", "2026-06-01T10:00",
    description="Cornetto + espresso",
    category="meal", location="Cafe Roma", cost=15.0,
))
plan.add_slot(1, slot(
    "Walk to Colosseum", "2026-06-01T10:00", "2026-06-01T11:00",
    category="transport",
))
plan.add_slot(1, slot(
    "Colosseum visit", "2026-06-01T11:00", "2026-06-01T15:00",
    description="Skip-the-line ticket",
    category="attraction", location="Colosseum", cost=24.0,
))
plan.add_slot(1, slot(
    "Party in Trastevere", "2026-06-01T19:00", "2026-06-02T01:00",
    category="leisure", location="Trastevere", cost=40.0,
))

# Day 2
plan.add_slot(2, slot(
    "Hike Appian Way", "2026-06-02T09:00", "2026-06-02T13:00",
    category="attraction", cost=0.0,
))
plan.add_slot(2, slot(
    "Lunch", "2026-06-02T13:30", "2026-06-02T15:00",
    category="meal", cost=22.0,
))

from IPython.display import Markdown
Markdown(plan.to_markdown())

# TravelPlan: Rome 2-day demo

| Day 1 — 2026-06-01 — Arrival | Day 2 — 2026-06-02 |
| --- | --- |
| **1. Breakfast** [meal] 08:00–10:00 @ Cafe Roma (€15.00) — Cornetto + espresso | **1. Hike Appian Way** [attraction] 09:00–13:00 (€0.00) |
| **2. Walk to Colosseum** [transport] 10:00–11:00 | **2. Lunch** [meal] 13:30–15:00 (€22.00) |
| **3. Colosseum visit** [attraction] 11:00–15:00 @ Colosseum (€24.00) — Skip-the-line ticket |  |
| **4. Party in Trastevere** [leisure] 2026-06-01 19:00–2026-06-02 01:00 @ Trastevere (€40.00) |  |

**Total estimated cost: €101.00** (Day 1: €79.00, Day 2: €22.00)

## 2. Edit the plan: insert, delete, add a third day

In [3]:
# Insert a chill slot at position 1 of day 2 (before the hike)
plan.insert_slot(2, 1, slot(
    "Slow morning coffee", "2026-06-02T08:00", "2026-06-02T09:00",
    category="meal", cost=4.5,
))

# Drop the walk-to-colosseum slot (it was position 2 on day 1)
removed = plan.delete_slot(1, 2)
print("removed:", removed.name)

# Add a third day
plan.add_day(label="Departure", calendar_date=date(2026, 6, 3))
plan.add_slot(3, slot(
    "Flight home", "2026-06-03T11:00", "2026-06-03T14:00",
    category="transport", cost=180.0,
))

Markdown(plan.to_markdown())

removed: Walk to Colosseum


# TravelPlan: Rome 2-day demo

| Day 1 — 2026-06-01 — Arrival | Day 2 — 2026-06-02 | Day 3 — 2026-06-03 — Departure |
| --- | --- | --- |
| **1. Breakfast** [meal] 08:00–10:00 @ Cafe Roma (€15.00) — Cornetto + espresso | **1. Slow morning coffee** [meal] 08:00–09:00 (€4.50) | **1. Flight home** [transport] 11:00–14:00 (€180.00) |
| **2. Colosseum visit** [attraction] 11:00–15:00 @ Colosseum (€24.00) — Skip-the-line ticket | **2. Hike Appian Way** [attraction] 09:00–13:00 (€0.00) |  |
| **3. Party in Trastevere** [leisure] 2026-06-01 19:00–2026-06-02 01:00 @ Trastevere (€40.00) | **3. Lunch** [meal] 13:30–15:00 (€22.00) |  |

**Total estimated cost: €285.50** (Day 1: €79.00, Day 2: €26.50, Day 3: €180.00)

## 3. Cost overview

In [4]:
summary = plan.cost_summary()
print("total:", summary.total)
print("per day:", summary.per_day)

total: 285.5
per day: {1: 79.0, 2: 26.5, 3: 180.0}


## 4. See the deterministic guards fire

Each cell below is expected to raise — useful to feel out what the agent will hit when it asks for something invalid.

In [5]:
try:
    plan.add_slot(1, slot(
        "Conflicting", "2026-06-01T12:00", "2026-06-01T14:00",
    ))
except SlotOverlapError as e:
    print("SlotOverlapError:", e)

SlotOverlapError: Slot 'Conflicting' [2026-06-01T12:00:00 – 2026-06-01T14:00:00] overlaps existing slot 'Colosseum visit' [2026-06-01T11:00:00 – 2026-06-01T15:00:00] on day 1


In [6]:
try:
    plan.get_day(99)
except DayNotFoundError as e:
    print("DayNotFoundError:", e)

DayNotFoundError: Day index 99 out of range (have 3 day(s))


In [ ]:
try:
    plan.delete_slot(1, 99)
except SlotNotFoundError as e:
    print("SlotNotFoundError:", e)

## 5. Serialize / round-trip

`TravelPlan` is a Pydantic model so it serializes cleanly — useful when the agent needs to pass state around.

In [7]:
blob = plan.model_dump(mode="json")
restored = TravelPlan.model_validate(blob)
assert restored.to_markdown() == plan.to_markdown()
print("round-trip OK — days:", [d.index for d in restored.days])

round-trip OK — days: [1, 2, 3]


In [8]:
blob

{'title': 'Rome 2-day demo',
 'days': [{'index': 1,
   'calendar_date': '2026-06-01',
   'label': 'Arrival',
   'slots': [{'name': 'Breakfast',
     'description': 'Cornetto + espresso',
     'start_time': '2026-06-01T08:00:00',
     'end_time': '2026-06-01T10:00:00',
     'category': 'meal',
     'location': 'Cafe Roma',
     'cost': 15.0,
     'notes': None},
    {'name': 'Colosseum visit',
     'description': 'Skip-the-line ticket',
     'start_time': '2026-06-01T11:00:00',
     'end_time': '2026-06-01T15:00:00',
     'category': 'attraction',
     'location': 'Colosseum',
     'cost': 24.0,
     'notes': None},
    {'name': 'Party in Trastevere',
     'description': '',
     'start_time': '2026-06-01T19:00:00',
     'end_time': '2026-06-02T01:00:00',
     'category': 'leisure',
     'location': 'Trastevere',
     'cost': 40.0,
     'notes': None}]},
  {'index': 2,
   'calendar_date': '2026-06-02',
   'label': None,
   'slots': [{'name': 'Slow morning coffee',
     'description': ''